# 01 - Exploratory Data Analysis (EDA)

This notebook performs exploratory data analysis on the resume dataset.

## Objectives
- Load and understand the dataset structure
- Explore key fields in resumes
- Identify data quality issues
- Select sample CVs for roasting

---

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

## 1. Load Dataset

In [ ]:
# Load the resume dataset
data_path = Path('../data/resume_data.csv')
df = pd.read_csv(data_path)

print(f"Dataset shape: {df.shape}")
print(f"Total resumes: {len(df)}")

## 2. Dataset Overview

In [ ]:
# Display column names
print("Columns in dataset:")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

In [ ]:
# Basic info
df.info()

In [ ]:
# Check for missing values
missing_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print("\nMissing values percentage:")
print(missing_pct[missing_pct > 0])

## 3. Key Resume Fields Analysis

In [ ]:
# Key fields for CV roasting
key_fields = [
    'career_objective',
    'skills',
    'educational_institution_name',
    'degree_names',
    'professional_company_names',
    'positions',
    'responsibilities'
]

print("Key fields availability:")
for field in key_fields:
    if field in df.columns:
        non_null = df[field].notna().sum()
        print(f"{field}: {non_null}/{len(df)} ({non_null/len(df)*100:.1f}%)")

## 4. Sample Resume Exploration

In [ ]:
# Function to display a resume nicely
def display_resume(idx):
    resume = df.iloc[idx]
    print("="*80)
    print(f"RESUME #{idx}")
    print("="*80)
    
    if pd.notna(resume.get('career_objective')):
        print(f"\n📝 CAREER OBJECTIVE:")
        print(resume['career_objective'])
    
    if pd.notna(resume.get('skills')):
        print(f"\n💼 SKILLS:")
        print(resume['skills'])
    
    if pd.notna(resume.get('educational_institution_name')):
        print(f"\n🎓 EDUCATION:")
        print(f"Institution: {resume['educational_institution_name']}")
        if pd.notna(resume.get('degree_names')):
            print(f"Degree: {resume['degree_names']}")
        if pd.notna(resume.get('major_field_of_studies')):
            print(f"Major: {resume['major_field_of_studies']}")
    
    if pd.notna(resume.get('professional_company_names')):
        print(f"\n🏢 WORK EXPERIENCE:")
        print(f"Company: {resume['professional_company_names']}")
        if pd.notna(resume.get('positions')):
            print(f"Position: {resume['positions']}")
        if pd.notna(resume.get('responsibilities')):
            print(f"Responsibilities:\n{resume['responsibilities']}")
    
    print("\n" + "="*80 + "\n")

# Display first resume
display_resume(0)

In [ ]:
# Display a few more samples
for idx in [1, 2, 3]:
    if idx < len(df):
        display_resume(idx)

## 5. Create Helper Function to Extract CV Text

This function will format a resume into readable text for the LLM to critique.

In [ ]:
def format_cv_for_llm(resume_row):
    """
    Format a resume row into a readable text for LLM processing.
    
    Args:
        resume_row: A pandas Series representing one resume
    
    Returns:
        str: Formatted CV text
    """
    cv_text = []
    
    # Career Objective
    if pd.notna(resume_row.get('career_objective')):
        cv_text.append(f"CAREER OBJECTIVE:\n{resume_row['career_objective']}")
    
    # Skills
    if pd.notna(resume_row.get('skills')):
        cv_text.append(f"\nSKILLS:\n{resume_row['skills']}")
    
    # Education
    education_parts = []
    if pd.notna(resume_row.get('educational_institution_name')):
        education_parts.append(f"Institution: {resume_row['educational_institution_name']}")
    if pd.notna(resume_row.get('degree_names')):
        education_parts.append(f"Degree: {resume_row['degree_names']}")
    if pd.notna(resume_row.get('major_field_of_studies')):
        education_parts.append(f"Major: {resume_row['major_field_of_studies']}")
    if pd.notna(resume_row.get('passing_years')):
        education_parts.append(f"Year: {resume_row['passing_years']}")
    
    if education_parts:
        cv_text.append(f"\nEDUCATION:\n" + "\n".join(education_parts))
    
    # Work Experience
    work_parts = []
    if pd.notna(resume_row.get('professional_company_names')):
        work_parts.append(f"Company: {resume_row['professional_company_names']}")
    if pd.notna(resume_row.get('positions')):
        work_parts.append(f"Position: {resume_row['positions']}")
    if pd.notna(resume_row.get('start_dates')):
        work_parts.append(f"Period: {resume_row['start_dates']}")
        if pd.notna(resume_row.get('end_dates')):
            work_parts.append(f" to {resume_row['end_dates']}")
    if pd.notna(resume_row.get('responsibilities')):
        work_parts.append(f"Responsibilities:\n{resume_row['responsibilities']}")
    
    if work_parts:
        cv_text.append(f"\nWORK EXPERIENCE:\n" + "\n".join(work_parts))
    
    # Languages
    if pd.notna(resume_row.get('languages')):
        cv_text.append(f"\nLANGUAGES:\n{resume_row['languages']}")
    
    # Certifications
    if pd.notna(resume_row.get('certification_skills')):
        cv_text.append(f"\nCERTIFICATIONS:\n{resume_row['certification_skills']}")
    
    return "\n".join(cv_text)

# Test the function
print("Formatted CV for LLM:")
print("="*80)
print(format_cv_for_llm(df.iloc[0]))
print("="*80)

## 6. Select Test CVs

We'll select 2 diverse CVs to test each roasting model.

In [ ]:
# Select test CVs - choosing ones with good data coverage
# We'll select CVs with different characteristics

# Find CVs with most complete information
df['completeness'] = df[key_fields].notna().sum(axis=1)
most_complete = df.nlargest(5, 'completeness')

print("Top 5 most complete CVs:")
print(most_complete[['career_objective', 'completeness']].head())

# Select 2 CVs for testing
test_cv_indices = [0, 1]  # You can change these

print(f"\nSelected test CVs: {test_cv_indices}")

# Save test CV indices for use in other notebooks
with open('../data/test_cv_indices.json', 'w') as f:
    json.dump({'indices': test_cv_indices}, f)

## 7. Summary Statistics

In [ ]:
print("📊 DATASET SUMMARY")
print("="*80)
print(f"Total Resumes: {len(df)}")
print(f"Total Features: {len(df.columns)}")
print(f"\nAverage Completeness: {df['completeness'].mean():.1f}/{len(key_fields)} fields")
print(f"Most Complete CV: {df['completeness'].max()}/{len(key_fields)} fields")
print(f"Least Complete CV: {df['completeness'].min()}/{len(key_fields)} fields")
print("\n✅ EDA Complete! Ready for CV roasting.")

---

## Next Steps

1. **02_gentle_roaster.ipynb** - Constructive feedback model
2. **03_medium_roaster.ipynb** - Balanced criticism model
3. **04_brutal_roaster.ipynb** - Savage roasting model
4. **05_evaluation_comparison.ipynb** - Compare all models